
# IS 445 — UFO Sightings: Shapes and Timing

This notebook creates two original Altair visualizations from the same UFO sightings dataset, read directly from a public URL.

### Visualization 1: Bar Chart
Ten most-reported UFO shapes.

### Visualization 2: Interactive Heatmap
Number of reported sightings by calendar month and hour, with a country dropdown.

**Instructions:** Run all cells after installing pandas, altair, and jupyter.


In [9]:
import pandas as pd
import altair as alt
from pathlib import Path

# Required: load the original dataset directly from a URL.
DATA_URL = (
    "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/"
    "is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"
)

# This file has no header row, so name its columns explicitly.
columns = [
    "date_sighted", "city", "state", "country", "shape", "duration",
    "duration_txt", "note", "date_reported", "latitude", "longitude"
]

ufo = pd.read_csv(DATA_URL, header=None, names=columns, low_memory=False)
print(f"Loaded {len(ufo):,} UFO reports from the URL.")
ufo.head()

Loaded 80,332 UFO reports from the URL.


,date_sighted,city,state,country,shape,duration,duration_txt,note,date_reported,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611



## Data Preparation

Missing or invalid dates are excluded from the heatmap, but not automatically from the shape bar chart.

Some source times use `24:00` and are normalized to `00:00` for hour-of-day analysis (a small source-time simplification).

The raw CSV is not copied into the website repository because it is loaded directly from its public URL.


In [10]:
# Clean categorical columns and date/time values before aggregation.
ufo["shape"] = ufo["shape"].astype("string").str.strip().str.lower()
ufo["country"] = ufo["country"].astype("string").str.strip().str.lower()

# Some source records use 24:00. Normalize that to 00:00 so we can
# analyze the hour of day. Those rare records stay on their original date.
cleaned_dates = (
    ufo["date_sighted"].astype("string")
    .str.replace("24:00", "00:00", regex=False)
)
ufo["sighting_time"] = pd.to_datetime(
    cleaned_dates, format="%m/%d/%Y %H:%M", errors="coerce"
)

print(f"Reports with usable dates: {ufo['sighting_time'].notna().sum():,}")
print(f"Reports with usable shapes: {ufo['shape'].notna().sum():,}")

Reports with usable dates: 80,332
Reports with usable shapes: 78,400



## Plot 1: Top 10 UFO Shapes

We group the UFO sighting records by `shape` and select the ten most frequently reported shapes.

We use a **quantitative encoding (Q)** for the number of sightings, represented by bar length and blue color intensity. The UFO shapes use a **nominal encoding (N)** because they are categories without a natural order.


In [11]:
# Chart 1: counts of the ten most frequently reported shapes.
shape_counts = (
    ufo.dropna(subset=["shape"])
       .groupby("shape")
       .size()
       .sort_values(ascending=False)
       .head(10)
       .reset_index(name="reports")
)

chart1 = (
    alt.Chart(shape_counts)
    .mark_bar()
    .encode(
        x=alt.X("reports:Q", title="Number of reported sightings"),
        y=alt.Y("shape:N", sort="-x", title="Reported shape"),
        color=alt.Color(
            "reports:Q", scale=alt.Scale(scheme="blues"), legend=None
        ),
        tooltip=[
            alt.Tooltip("shape:N", title="Shape"),
            alt.Tooltip("reports:Q", title="Reports", format=",d")
        ]
    )
    .properties(title="Top 10 Reported UFO Shapes", width=600, height=350)
)
chart1

alt.Chart(...)


## Plot 2: UFO Sightings by Month and Hour

This heatmap shows the number of reported UFO sightings across different months and hours of the day.

We extract the **month** and **hour** from cleaned timestamps and group the records by each month-hour combination to calculate the total number of sightings.

We use **ordinal encodings (O)** for months and hours because they follow a natural order, and a **quantitative encoding (Q)** for the number of sightings, represented by color intensity.

### Interactivity

An interactive **dropdown menu** allows users to filter the heatmap by country or view all countries together.

This makes it easier to compare reporting patterns across different countries and identify when UFO sightings are most frequently reported.


In [12]:
# Chart 2: group valid reports by month and hour of the sighting.
valid = ufo.dropna(subset=["sighting_time"]).copy()
valid["month"] = valid["sighting_time"].dt.strftime("%b")
valid["hour"] = valid["sighting_time"].dt.hour

# "All countries" includes every record with a valid date, even if the
# country field is missing. Additional options use reported country codes.
all_counts = (
    valid.groupby(["month", "hour"])
         .size()
         .reset_index(name="reports")
)
all_counts["region"] = "All countries"

country_names = {
    "us": "United States", "ca": "Canada", "gb": "United Kingdom",
    "au": "Australia"
}
country_rows = valid[valid["country"].isin(country_names)].copy()
country_rows["region"] = country_rows["country"].map(country_names)
country_counts = (
    country_rows.groupby(["region", "month", "hour"])
                .size()
                .reset_index(name="reports")
)

heatmap_data = pd.concat(
    [all_counts, country_counts], ignore_index=True
)
options = ["All countries"] + [
    label for label in country_names.values()
    if label in country_counts["region"].unique()
]

country_select = alt.param(
    name="selected_country",
    value="All countries",
    bind=alt.binding_select(options=options, name="Country: ")
)

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

chart2 = (
    alt.Chart(heatmap_data)
    .mark_rect()
    .encode(
        x=alt.X("hour:O", sort=list(range(24)), title="Hour of sighting (24-hour clock)"),
        y=alt.Y("month:N", sort=month_order, title="Month"),
        color=alt.Color(
            "reports:Q", scale=alt.Scale(scheme="blues"),
            title="Number of reports"
        ),
        tooltip=[
            alt.Tooltip("region:N", title="Country selection"),
            alt.Tooltip("month:N", title="Month"),
            alt.Tooltip("hour:O", title="Hour"),
            alt.Tooltip("reports:Q", title="Reports", format=",d")
        ]
    )
    .add_params(country_select)
    .transform_filter(alt.datum.region == country_select)
    .properties(title="UFO Reports by Month and Hour", width=650, height=340)
)
chart2

alt.Chart(...)


## Export Interactive HTML to the Jekyll Website

This section exports both Altair visualizations as `.html` files so they can be displayed on our GitHub Pages website.

The exported HTML files preserve the interactive features, including the country dropdown menu in Plot 2.

### Files to Upload to GitHub

The following files must be committed and pushed to the GitHub repository:

1. **Two HTML files** — The exported visualizations.
2. **Python Notebook (.ipynb)** — The code used to create the charts.
3. **Jekyll Post (.md)** — The webpage containing both visualizations and their explanations.

All files must be uploaded to GitHub so the visualizations can be viewed publicly and graded.


In [13]:
# Locate the existing Jekyll repository. This works whether the notebook
# is launched from the repository root or from python_notebooks/.
working_dir = Path.cwd()
if (working_dir / "_config.yml").exists():
    repo_root = working_dir
elif (working_dir.parent / "_config.yml").exists():
    repo_root = working_dir.parent
else:
    raise FileNotFoundError(
        "Open and run this notebook from your koyouwu2.github.io repository "
        "or its python_notebooks folder."
    )

plot_dir = repo_root / "assets" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
chart1.save(str(plot_dir / "ufo_shapes.html"))
chart2.save(str(plot_dir / "ufo_time_heatmap.html"))

print("Export complete:")
print(plot_dir / "ufo_shapes.html")
print(plot_dir / "ufo_time_heatmap.html")
print("Commit these HTML files, the notebook, and the Jekyll post to GitHub.")

Export complete:
c:\Users\kathy\OneDrive\桌面\koyouwu2.github.io\koyouwu2.github.io\assets\plots\ufo_shapes.html
c:\Users\kathy\OneDrive\桌面\koyouwu2.github.io\koyouwu2.github.io\assets\plots\ufo_time_heatmap.html
Commit these HTML files, the notebook, and the Jekyll post to GitHub.
